In [1]:
import os
import pickle
import matplotlib.pyplot as plt
import torch
import numpy as np
from nnfabrik.builder import get_data, get_trainer

from model import stacked_core_full_gauss_readout
from trainer import standard_trainer
from sensorium.utility.scores import get_correlations

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

from mog_vae.model import MoGVAE

device = "cuda:7"
torch.cuda.set_device(device)

In [2]:
basepath = "/srv/user/polina/sensorium/sensorium/notebooks/data/"

# as filenames, we'll select all 7 datasets
filenames = [
    os.path.join(basepath, file) for file in os.listdir(basepath) if ".zip" in file
]


dataset_fn = "sensorium.datasets.static_loaders"
dataset_config = {
    "paths": filenames,
    "normalize": True,
    "include_behavior": True,
    "include_eye_position": True,
    "batch_size": 128,
    "scale": 0.25,
}

dataloaders = get_data(dataset_fn, dataset_config)
data_keys = list(dataloaders['train'].keys())

In [3]:
model_config = {
    "pad_input": False,
    "stack": -1,
    "layers": 4,
    "input_kern": 9,
    "gamma_input": 6.3831,
    "gamma_readout": 0.0076,
    "hidden_kern": 7,
    "hidden_channels": 64,
    "depth_separable": True,
    "grid_mean_predictor": {
        "type": "cortex",
        "input_dimensions": 2,
        "hidden_layers": 1,
        "hidden_features": 30,
        "final_tanh": True,
    },
    "init_sigma": 0.1,
    "init_mu_range": 0.3,
    "gauss_type": "full",
    "shifter": True, 
    "autoencoder": None,
}

trainer_config = {
    'max_iter': 200,
    'verbose': False,
    'lr_decay_steps': 4,
    'avg_loss': False,
    'lr_init': 0.009,
    'device': device, 
}

In [4]:
model_seeds = [0, 1, 2]

In [5]:
from torch.nn import functional as F

def recon_loss_fn(x, x_rec):
    loss = (x - x_rec).pow(2).sum(-1)
    return loss.sum()

def log_normal(x, mu, var, eps=1e-8):
    var = var + eps
    return -0.5 * (np.log(2.0 * np.pi) + torch.log(var) + (x - mu).pow(2) / var).sum(dim=-1)

def gaussian_loss_fn(z, z_mu, z_var, z_mu_prior, z_var_prior):
    loss = log_normal(z, z_mu, z_var) - log_normal(z, z_mu_prior, z_var_prior)
    return loss.sum()

def entropy_loss_fn(logits, probs):
    log_q = F.log_softmax(logits, dim=-1)
    loss = (probs * log_q).sum(dim=-1)
    return loss.sum()

In [6]:
def train_autoenc(autoencoder, features, w_rec, w_gauss, w_entr, temp_init, temp_decay, temp_min, hard_softmax):
    optim = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, factor=0.3, patience=100)

    epochs = int(1e15)
    min_lr = 1e-5

    for epoch in range(epochs):
        temperature = max(temp_init * np.exp(-temp_decay * epoch), temp_min)
        model_out = autoencoder(features, return_params=True, temperature=temperature, hard=hard_softmax)
        
        recon = model_out['x_rec']
        recon_loss = recon_loss_fn(features, recon)

        z = model_out['z']
        z_mu, z_var = model_out['mu'], model_out['var']
        z_mu_prior, z_var_prior = model_out['y_mu'], model_out['y_var']
        gaussian_loss = gaussian_loss_fn(z, z_mu, z_var, z_mu_prior, z_var_prior)

        logits, probs = model_out['logits'], model_out['probs']
        entropy_loss = entropy_loss_fn(logits, probs)

        loss = w_rec * recon_loss + w_gauss * gaussian_loss + w_entr * entropy_loss

        optim.zero_grad()
        loss.backward()
        optim.step()
        scheduler.step(loss.item())
        if epoch % 100 == 0:
            lr = optim.param_groups[0]['lr']
            if lr < min_lr:
                break

            print(recon_loss.item(), gaussian_loss.item(), entropy_loss.item())
            print(lr, temperature)

In [7]:
# Autoencoder configs for different latent dims that result in less than 1% performance drop
autoencoder_configs = [
    { 'latent_dim': 32, 'hidden_dims': 512, 'hidden_layers': 3, }, 
    { 'latent_dim': 16, 'hidden_dims': 512, 'hidden_layers': 5, }, 
    { 'latent_dim': 8, 'hidden_dims': 512, 'hidden_layers': 7, }, 
    { 'latent_dim': 4, 'hidden_dims': 1024, 'hidden_layers': 7, }, 
]

In [8]:
checkpoint = torch.load(f'checkpoints/base/model_weights{0}.pth')

# Extract features from all mice
features = []
for data_key in data_keys:
    features_single_mouse = checkpoint[f'readout.{data_key}._features']
    features_single_mouse = features_single_mouse.squeeze().permute(1, 0).detach()
    features.append(features_single_mouse)
features = torch.cat(features)

In [52]:
mog_vae = MoGVAE(64, 32, hidden_layers=3, hidden_dims=512, batch_norm=True, nonlinearity='GELU', num_components=10)
mog_vae.to(device)

MoGVAE(
  (inference): InferenceNet(
    (qyx_layers): ModuleList(
      (0): Linear(in_features=64, out_features=512, bias=True)
      (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Linear(in_features=512, out_features=512, bias=True)
      (4): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): GELU(approximate='none')
      (6): Linear(in_features=512, out_features=512, bias=True)
      (7): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): GELU(approximate='none')
      (9): GumbelSoftmax(
        (logits): Linear(in_features=512, out_features=10, bias=True)
      )
    )
    (qzyx_layers): ModuleList(
      (0): Linear(in_features=74, out_features=512, bias=True)
      (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Linear(in_featu

In [53]:
train_autoenc(mog_vae, features, w_rec=200, w_gauss=1, w_entr=2, temp_init=1, temp_decay=5e-4, temp_min=0.1, hard_softmax=True)

2857706.0 1249697.25 -108634.3671875
0.001 1.0


138338.96875 3295162.0 -65271.609375
0.001 0.951229424500714
107311.21875 3512557.0 -62705.0
0.001 0.9048374180359595
96783.8125 3551927.0 -76860.09375
0.001 0.8607079764250578
89240.1640625 3563201.0 -88158.421875
0.001 0.8187307530779818
84333.2578125 3467153.75 -92028.8203125
0.001 0.7788007830714049
80050.40625 3488953.0 -92858.625
0.001 0.7408182206817179
76649.265625 3504967.5 -92948.1640625
0.001 0.7046880897187134
74651.21875 3524003.5 -92672.3125
0.001 0.6703200460356393
71135.125 3540267.75 -92774.5
0.001 0.6376281516217733
68094.8203125 3555494.0 -91843.3203125
0.001 0.6065306597126334
65392.4296875 3576476.0 -90686.3125
0.001 0.5769498103804866
62954.6015625 3599397.5 -89446.640625
0.001 0.5488116360940264
60850.41015625 3607046.75 -88625.234375
0.001 0.522045776761016
58492.59375 3634263.0 -87027.390625
0.001 0.49658530379140947
56646.859375 3651655.0 -86437.9375
0.001 0.4723665527410147
55175.5 3657829.75 -85374.265625
0.001 0.44932896411722156
53436.09375 3675964.5 -8444

In [54]:
checkpoint = torch.load(f'checkpoints/base/model_weights{0}.pth')
model = stacked_core_full_gauss_readout(dataloaders, 0, **model_config)
model.load_state_dict(checkpoint)
model.eval()

validation_score = get_correlations(
    model, dataloaders["validation"], device=device, as_dict=False, per_neuron=False
)
print('base', validation_score)

mog_vae.eval()

for data_key in data_keys:
    model.readout[data_key].autoencoder = mog_vae

validation_score = get_correlations(
    model, dataloaders["validation"], device=device, as_dict=False, per_neuron=False
)
print(validation_score)

base 0.38690987
0.367075


In [37]:
ys = torch.eye(10)

mu, var = mog_vae.generative.pzy(ys)

In [55]:
mog_vae.to(device)

MoGVAE(
  (inference): InferenceNet(
    (qyx_layers): ModuleList(
      (0): Linear(in_features=64, out_features=512, bias=True)
      (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Linear(in_features=512, out_features=512, bias=True)
      (4): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): GELU(approximate='none')
      (6): Linear(in_features=512, out_features=512, bias=True)
      (7): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): GELU(approximate='none')
      (9): GumbelSoftmax(
        (logits): Linear(in_features=512, out_features=10, bias=True)
      )
    )
    (qzyx_layers): ModuleList(
      (0): Linear(in_features=74, out_features=512, bias=True)
      (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Linear(in_featu

In [56]:
# Cluster attribution numbers
logits, probs, y = mog_vae.inference.qyx(features, temperature=1., hard=False)
y = y.detach().cpu().numpy()
print(y[0])
y = y.argmax(axis=1)
[(y == i).sum(axis=0) for i in range(10)]

[3.4005299e-02 3.0333156e-02 1.3434507e-15 1.9709298e-01 8.4826052e-02
 7.5992920e-02 1.9728988e-14 2.7492815e-01 2.0877367e-01 9.4047755e-02]


[np.int64(5167),
 np.int64(3395),
 np.int64(7628),
 np.int64(5405),
 np.int64(5369),
 np.int64(5261),
 np.int64(6614),
 np.int64(5172),
 np.int64(5378),
 np.int64(5180)]

In [ ]:
# Distance matrix of means
(mu[:, None, :] - mu[None, :, :]).pow(2).sum(dim=-1).sqrt()

tensor([[ 0.0000,  3.7800,  1.4089, 10.3518,  1.9073],
        [ 3.7800,  0.0000,  3.4168,  7.2538,  4.1340],
        [ 1.4089,  3.4168,  0.0000, 10.1503,  2.2470],
        [10.3518,  7.2538, 10.1503,  0.0000, 10.6109],
        [ 1.9073,  4.1340,  2.2470, 10.6109,  0.0000]],
       grad_fn=<SqrtBackward0>)

In [ ]:
# Train all autoencoders for each model

all_features = []

for model_seed in model_seeds:
    checkpoint = torch.load(f'checkpoints/base/model_weights{model_seed}.pth')

    # Extract features from all mice
    features = []
    for data_key in data_keys:
        features_single_mouse = checkpoint[f'readout.{data_key}._features']
        features_single_mouse = features_single_mouse.squeeze().permute(1, 0).detach()
        features.append(features_single_mouse)
    features = torch.cat(features)

    feature_dict = {
        'base': features.detach().cpu().numpy()
    }

    for autoencoder_config in autoencoder_configs:
        autoencoder = VAE(
            64, batch_norm=True, nonlinearity='GELU', **autoencoder_config
        )
        autoencoder.load_state_dict(
            torch.load(f'checkpoints/autoencoder/model_weights{model_seed}_{autoencoder_config['latent_dim']}.pth')
        )
        autoencoder.to(device)
        #train_autoenc(autoencoder, features)
        #torch.save(
        #    autoencoder.state_dict(), 
        #    f'checkpoints/autoencoder/model_weights{model_seed}_{autoencoder_config['latent_dim']}.pth'
        #)

        latent_features = autoencoder.encode(features)[0].detach().cpu().numpy()
        feature_dict[autoencoder_config['latent_dim']] = latent_features

    all_features.append(feature_dict)

    with open('all_features.pkl', 'wb') as f:
        pickle.dump(all_features, f)

NameError: name 'VAE' is not defined